# 🎤 Comedian Emotion Scorer — Training on Kaggle
**Dataset:** Human Face Emotions (9 400 images, 8 classes)  
**Modèles:** YOLOv8n · YOLOv8s · YOLOv8m  
**GPU:** T4/P100 Kaggle (gratuit)

## 1. Installation des dépendances

In [ ]:
!pip install ultralytics roboflow -q

## 2. Vérification GPU

In [ ]:
import torch
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print(f'Device utilisé: {DEVICE}')

## 3. Téléchargement du dataset depuis Roboflow
⚠️ **Remplace `YOUR_API_KEY` par ta vraie clé Roboflow** (Settings → API Key sur roboflow.com)

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "YOUR_API_KEY"  # <-- remplace ici

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("louiss-workspace-jmpoo").project("human-face-emotions-avush")
version = project.version(1)
dataset = version.download("yolov8")

DATASET_YAML = dataset.location + "/data.yaml"
print(f"Dataset prêt: {DATASET_YAML}")

## 4. Configuration de l'entraînement

In [ ]:
MODELS = [
    {"name": "YOLOv8n", "weights": "yolov8n.pt", "desc": "Nano - ultra-rapide"},
    {"name": "YOLOv8s", "weights": "yolov8s.pt", "desc": "Small - équilibre vitesse/précision"},
    {"name": "YOLOv8m", "weights": "yolov8m.pt", "desc": "Medium - meilleure précision"},
]

TRAIN_CONFIG = {
    "epochs": 50,
    "imgsz": 640,
    "batch": 16,
    "patience": 15,
    "workers": 4,
    "project": "runs/emotion_detection",
    "exist_ok": True,
    "device": DEVICE,
}

print("Config prête:", TRAIN_CONFIG)

## 5. Entraînement des 3 modèles

In [ ]:
import json
import time
from pathlib import Path
from ultralytics import YOLO

all_results = []

for cfg in MODELS:
    print(f"\n{'='*60}")
    print(f"Entraînement: {cfg['name']} — {cfg['desc']}")
    print(f"{'='*60}")

    t0 = time.time()
    model = YOLO(cfg["weights"])
    results = model.train(data=DATASET_YAML, name=cfg["name"], **TRAIN_CONFIG)
    elapsed = time.time() - t0

    metrics = {
        "model_name": cfg["name"],
        "description": cfg["desc"],
        "training_time_formatted": f"{elapsed/3600:.1f}h {(elapsed%3600)/60:.0f}m",
        "training_time_seconds": round(elapsed, 2),
        "mAP50": float(results.results_dict.get("metrics/mAP50(B)", 0)),
        "mAP50_95": float(results.results_dict.get("metrics/mAP50-95(B)", 0)),
        "precision": float(results.results_dict.get("metrics/precision(B)", 0)),
        "recall": float(results.results_dict.get("metrics/recall(B)", 0)),
        "best_model_path": f"runs/emotion_detection/{cfg['name']}/weights/best.pt",
    }
    all_results.append(metrics)

    print(f"✅ {cfg['name']} — mAP50: {metrics['mAP50']:.4f} | Temps: {metrics['training_time_formatted']}")

with open("training_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("\n📊 Résultats sauvegardés dans training_results.json")

## 6. Comparaison des modèles

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

valid = sorted(all_results, key=lambda x: x.get("mAP50_95", 0), reverse=True)

print("\n" + "="*70)
print("COMPARAISON DES MODÈLES")
print("="*70)
print(f"{'Rang':<6} {'Modèle':<12} {'mAP50':>8} {'mAP50-95':>10} {'Précision':>10} {'Rappel':>8} {'Temps':>10}")
print("-"*70)
medals = ["🥇", "🥈", "🥉"]
for i, r in enumerate(valid):
    print(f"{medals[i]:<6} {r['model_name']:<12} {r.get('mAP50',0):>8.4f} {r.get('mAP50_95',0):>10.4f} {r.get('precision',0):>10.4f} {r.get('recall',0):>8.4f} {r.get('training_time_formatted','N/A'):>10}")
print("-"*70)
print(f"\n🏆 Meilleur modèle: {valid[0]['model_name']} (mAP50-95: {valid[0].get('mAP50_95',0):.4f})")

In [ ]:
# Graphique comparatif
names = [r["model_name"] for r in valid]
metrics_plot = {
    "mAP50":     [r.get("mAP50", 0) for r in valid],
    "mAP50-95":  [r.get("mAP50_95", 0) for r in valid],
    "Précision": [r.get("precision", 0) for r in valid],
    "Rappel":    [r.get("recall", 0) for r in valid],
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Comparaison des Modèles — Détection d'Émotions Faciales", fontsize=15, fontweight="bold")

x = np.arange(len(names))
width = 0.2
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
ax1 = axes[0]
for i, (metric, vals) in enumerate(metrics_plot.items()):
    ax1.bar(x + i * width, vals, width, label=metric, color=colors[i], alpha=0.85)
ax1.set_xticks(x + width * 1.5)
ax1.set_xticklabels(names)
ax1.set_ylim(0, 1.1)
ax1.set_title("Métriques par Modèle")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

ax2 = plt.subplot(122, polar=True)
categories = list(metrics_plot.keys())
N = len(categories)
angles = [n / N * 2 * np.pi for n in range(N)]
angles += angles[:1]
ax2.set_theta_offset(np.pi / 2)
ax2.set_theta_direction(-1)
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(categories)
ax2.set_ylim(0, 1)
ax2.set_title("Radar des Performances", pad=20)
radar_colors = ["#e41a1c", "#377eb8", "#4daf4a"]
for i, r in enumerate(valid):
    vals = [r.get("mAP50",0), r.get("mAP50_95",0), r.get("precision",0), r.get("recall",0)]
    vals += vals[:1]
    ax2.plot(angles, vals, "o-", linewidth=2, label=r["model_name"], color=radar_colors[i])
    ax2.fill(angles, vals, alpha=0.1, color=radar_colors[i])
ax2.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("📈 Graphique sauvegardé: model_comparison.png")

## 7. Téléchargement des résultats

In [ ]:
import shutil, os

# Zipper les poids des 3 modèles + résultats
os.makedirs("export", exist_ok=True)
shutil.copy("training_results.json", "export/")
shutil.copy("model_comparison.png", "export/")

for r in valid:
    best_pt = r["best_model_path"]
    if os.path.exists(best_pt):
        dest = f"export/{r['model_name']}_best.pt"
        shutil.copy(best_pt, dest)
        size_mb = os.path.getsize(dest) / 1e6
        print(f"✅ {dest} ({size_mb:.1f} MB)")
    else:
        print(f"⚠️  Poids non trouvés: {best_pt}")

shutil.make_archive("emotion_scorer_results", "zip", "export")
print("\n📦 Archive prête: emotion_scorer_results.zip")
print("👉 Télécharge ce fichier depuis le panneau Files de Kaggle (à droite)")

## 8. Commande webcam (à exécuter en local après téléchargement)

```bash
# Remplace YOLOv8s_best.pt par le meilleur modèle indiqué ci-dessus
python predict_emotions.py --model YOLOv8s_best.pt --source 0
```